# Synthetic repeated-facies data

This notebook is the single source of the synthetic dataset used by the benchmark,
sensitivity, coordinate-scaling, Geary-permutation, and lambda experiments.
Run all cells to generate and inspect the data; no embeddings or benchmarks run here.

The fixed seed is **42**. A jittered **48 × 48** grid gives **2,304 samples**,
with sixteen **12 × 12** patches and four equally represented facies. The rule
`facies = (tile_x + tile_y) % 4` repeats each facies in four disconnected patches.
Coordinates are in metres, with nominal spacing 25 m and about 1 m jitter.

The 50 features consist of 10 noisy facies signals, 20 dependent nonlinear
confounders with spatially shuffled latent values, and 20 independent noise variables.
Truth and patch labels are metadata for evaluation, never feature inputs.
The returned latent values support diagnostics only. Each experiment applies its
own preprocessing to the raw data and coordinates.

## Reuse in another notebook

Add the repository's `code/` directory to `sys.path`, then call:

```python
from synthetic_data import make_synthetic_repeated_facies, FEATURE_COLUMNS, XY_COLUMNS
data, latent = make_synthetic_repeated_facies(random_state=42)
```

`code/synthetic_data.py` loads only the cells tagged `synthetic-data-definitions`.
It does not run the inspection cells or write files. No prior execution, generated
CSV, Jupyter-specific loader, or compiled SpaSNE executable is required.
The original generation algorithm and random-number draw order are preserved.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd


## Data configuration


In [ ]:
SEED = 42
GRID_SIDE = 48
TILES_PER_SIDE = 4
TILE_SIDE = GRID_SIDE // TILES_PER_SIDE
N_SAMPLES = GRID_SIDE**2
N_FEATURES = 50
KNOWN_CLASS_COUNT = 4
XY_COLUMNS = ["x_m", "y_m"]
LABEL_COLUMN = "truth_facies"
SPATIAL_FEATURES = [f"spatial_{index:02d}" for index in range(1, 11)]
CONFOUNDER_FEATURES = [f"confounder_{index:02d}" for index in range(1, 21)]
NOISE_FEATURES = [f"noise_{index:02d}" for index in range(1, 21)]
FEATURE_COLUMNS = SPATIAL_FEATURES + CONFOUNDER_FEATURES + NOISE_FEATURES
METADATA_COLUMNS = [
    "sample_id",
    *XY_COLUMNS,
    LABEL_COLUMN,
    "facies_id",
    "patch_id",
]
EXPECTED_LABELS = {
    0: "Facies A",
    1: "Facies B",
    2: "Facies C",
    3: "Facies D",
}


## Generator

Uses a local NumPy random generator; calling it does not change global RNG state.


In [ ]:
def make_synthetic_repeated_facies(
    random_state: int = SEED,
) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    """Create the repeated-facies benchmark and its plotting-only latent values."""
    # 1. Use one local random generator so the complete dataset is reproducible.
    rng = np.random.default_rng(random_state)

    # 2. Create a 48 x 48 sampling grid, then add about 1 m of coordinate jitter.
    #    Adjacent nominal samples remain 25 m apart, but the layout is not perfectly regular.
    grid_axis = np.arange(GRID_SIDE)
    x_grid, y_grid = np.meshgrid(grid_axis, grid_axis)
    x_index = x_grid.ravel().astype(np.int64)
    y_index = y_grid.ravel().astype(np.int64)
    coordinate_indices = np.column_stack([x_index, y_index]).astype(np.float64)
    coordinate_jitter = rng.normal(0.0, 0.04, size=coordinate_indices.shape)
    coordinates = 25.0 * (coordinate_indices + coordinate_jitter)

    # 3. Divide the grid into sixteen 12 x 12 physical patches.
    tile_x = x_index // TILE_SIDE
    tile_y = y_index // TILE_SIDE
    patch_id = tile_y * TILES_PER_SIDE + tile_x

    # Apply a Latin-square rule. Each facies occurs in four disconnected patches.
    facies_id = (tile_x + tile_y) % KNOWN_CLASS_COUNT

    # 4. Give every facies a balanced three-value signature.
    #    Any two facies differ in exactly two of the three signature values.
    facies_prototype = np.array(
        [
            [-1.0, -1.0, -1.0],  # Facies A
            [-1.0, +1.0, +1.0],  # Facies B
            [+1.0, -1.0, +1.0],  # Facies C
            [+1.0, +1.0, -1.0],  # Facies D
        ],
        dtype=np.float64,
    )

    # Build ten answer-bearing features from the three-value signatures.
    # Features cycle through signature positions 0, 1, 2 while their noise increases.
    spatial_noise_sigma = np.linspace(0.15, 0.95, 10)
    spatial_columns = []
    for feature_index, noise_sigma in enumerate(spatial_noise_sigma):
        prototype_position = feature_index % facies_prototype.shape[1]
        facies_signal = facies_prototype[facies_id, prototype_position]
        measurement_noise = noise_sigma * rng.normal(size=N_SAMPLES)
        spatial_columns.append(facies_signal + measurement_noise)
    spatial_block = np.column_stack(spatial_columns)

    # 5. Draw nuisance values independently for each sample. They have no intended
    #    relationship with facies or location, but their nonlinear structure is strong.
    nuisance_theta = rng.uniform(-np.pi, np.pi, size=N_SAMPLES)
    nuisance_q = rng.normal(size=N_SAMPLES)
    nuisance_radius = 1.0 + 0.18 * rng.normal(size=N_SAMPLES)

    # Name repeated transformations so the twenty confounders are easier to read.
    cos_theta = np.cos(nuisance_theta)
    sin_theta = np.sin(nuisance_theta)
    tanh_q = np.tanh(nuisance_q)
    centered_q_squared = nuisance_q**2 - 1.0

    confounder_basis = np.column_stack(
        [
            # A noisy circular manifold and its second and third harmonics.
            nuisance_radius * cos_theta,
            nuisance_radius * sin_theta,
            np.cos(2.0 * nuisance_theta),
            np.sin(2.0 * nuisance_theta),
            np.cos(3.0 * nuisance_theta),
            np.sin(3.0 * nuisance_theta),
            # Saturating, polynomial, and periodic transforms of q.
            tanh_q,
            centered_q_squared,
            np.sin(1.5 * nuisance_q),
            np.cos(1.5 * nuisance_q),
            # Additive mixtures of the angle and q structures.
            cos_theta + 0.55 * tanh_q,
            sin_theta - 0.55 * tanh_q,
            np.cos(2.0 * nuisance_theta) + 0.35 * centered_q_squared,
            np.sin(2.0 * nuisance_theta) - 0.35 * centered_q_squared,
            # Multiplicative interactions between the angle and q structures.
            tanh_q * cos_theta,
            tanh_q * sin_theta,
            centered_q_squared * cos_theta,
            centered_q_squared * sin_theta,
            # Two final nonlinear combinations.
            np.sin(nuisance_theta + nuisance_q),
            np.cos(nuisance_theta - nuisance_q),
        ]
    )
    confounder_noise = rng.normal(0.0, 0.12, size=confounder_basis.shape)
    confounder_block = confounder_basis + confounder_noise

    # 6. Add twenty completely independent distractor features.
    independent_block = rng.normal(size=(N_SAMPLES, 20))

    # 7. Join the three feature blocks in their documented 10 + 20 + 20 order.
    feature_matrix = np.column_stack(
        [spatial_block, confounder_block, independent_block]
    )
    assert feature_matrix.shape == (N_SAMPLES, N_FEATURES)

    # Keep labels and locations as metadata; they are not part of FEATURE_COLUMNS.
    metadata = pd.DataFrame(
        {
            "sample_id": [f"S{index:04d}" for index in range(N_SAMPLES)],
            "x_m": coordinates[:, 0],
            "y_m": coordinates[:, 1],
            LABEL_COLUMN: [EXPECTED_LABELS[value] for value in facies_id],
            "facies_id": facies_id,
            "patch_id": patch_id,
        }
    )
    feature_data = pd.DataFrame(feature_matrix, columns=FEATURE_COLUMNS)
    generated = pd.concat([metadata, feature_data], axis=1)

    # Return latent values only for diagnostics and plotting.
    latent = {
        "nuisance_theta": nuisance_theta,
        "spatial_noise_sigma": spatial_noise_sigma,
    }
    return generated, latent


## Generate and check the data


In [ ]:
data, latent = make_synthetic_repeated_facies()
assert data.shape == (N_SAMPLES, len(METADATA_COLUMNS) + N_FEATURES)
assert data.columns.tolist() == METADATA_COLUMNS + FEATURE_COLUMNS
assert data['sample_id'].is_unique
assert not data.isna().any().any()
assert np.isfinite(data[FEATURE_COLUMNS + XY_COLUMNS].to_numpy()).all()
assert data['patch_id'].nunique() == 16
assert data.groupby('facies_id').size().eq(N_SAMPLES // KNOWN_CLASS_COUNT).all()
display(data.head())
display(data.groupby(['facies_id', LABEL_COLUMN]).size().rename('samples').to_frame())
print(f'{len(data):,} samples; {len(FEATURE_COLUMNS)} features; {data.patch_id.nunique()} patches')
